In [71]:
!python -m pip install -e ..

Obtaining file:///Users/dkoffical/Documents/GitHub/cs321m_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for torch_measure (pyproject.toml) ... done
  Created wheel for torch_measure: filename=torch_measure-0.1.1.dev61-0.editable-py3-none-any.whl size=3611 sha256=f8ea770a04b004bbbc080cf594327946d74ec2d37f6aea1ea46439e2be75dbda
  Stored in directory: /private/var/folders/qv/knpkl_fn1dx05nr8ny_7hnwm0000gn/T/pip-ephem-wheel-cache-phupm_0c/wheels/db/9f/66/061ff46a269df88e4649b7b397f9a65a8676ab6ac2ce02b09e
Successfully built torch_measure
  Attempting uninstall: torch_measure
    Found existing installation: torch_measure 0.1.1.dev61
    Uninstalling torch_measure-0.1.1.dev61:
      Successfully uninstalled torch_measure-0.1.1.dev61


In [72]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cpu


### Load a response matrix

Choose one matrix preset by changing `KFACTOR_MATRIX`. Rows are models/subjects, columns are items, and values are the observed score/correctness used by the factor model.


In [ ]:
import pandas as pd
import torch
from torch_measure.data import ResponseMatrix

MATRIX_PRESETS = {
    "mmlu_solver": {
        "prefix": "mmlu_pro_solver",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_solver",
        "item_content_field": "source",
        "item_id_field": "pair_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed/missing",
    },
    "mmlu_judging": {
        "prefix": "mmlu_pro_judging",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_judging",
        "item_content_field": "source",
        "item_id_field": "pair_id/order",
        "value": "judge score/correctness; NaN=missing",
    },
    "safety_all_attacks": {
        "prefix": "safety_all_attacks",
        "dir": "../benchmarks/safety/final_solver/response_matrices",
        "benchmark_id": "safety_all_attacks",
        "item_content_field": "source",
        "item_id_field": "attack_family:input_index",
        "value": "score: 1.0=harmful/successful, 0.0=not harmful/unsuccessful, NaN=missing",
    },
    "harmmetric_eval": {
        "prefix": "harmmetric_eval",
        "dir": "../benchmarks/HarmMetric_Eval/response_matrices",
        "benchmark_id": "harmmetric_eval",
        "item_content_field": "source",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
    "kudge_challenge": {
        "prefix": "kudge_challenge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_challenge_easy_hard/response_matrices",
        "benchmark_id": "kudge_challenge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
    "kudge_judge": {
        "prefix": "kudge_judge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_judge_easy_hard/response_matrices",
        "benchmark_id": "kudge_judge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
}

KFACTOR_MATRIX = "safety_all_attacks"  # choose from MATRIX_PRESETS
config = MATRIX_PRESETS[KFACTOR_MATRIX]
matrix_dir = config["dir"]
prefix = config["prefix"]

matrix_path = f"{matrix_dir}/{prefix}_response_matrix.csv"
subject_meta_path = f"{matrix_dir}/{prefix}_subject_metadata.csv"
item_meta_path = f"{matrix_dir}/{prefix}_item_metadata.csv"

print(f"Loading {KFACTOR_MATRIX}")
print(matrix_path)

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

def enrich_kudge_item_metadata(item_meta):
    """Add prompt/source text from the original KUDGE dataset when available."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("datasets is not installed; item metadata will only include response-matrix fields.")
        return item_meta

    try:
        ds = load_dataset("amphora/kudge-challenge", split="train")
    except Exception as exc:
        print(f"Could not load amphora/kudge-challenge; item metadata will only include response-matrix fields: {exc}")
        return item_meta

    source_rows = []
    for row in ds:
        if row.get("subset") not in {"Korean-Easy", "Korean-Hard"}:
            continue
        source_rows.append({
            "item_id": str(row.get("id")),
            "prompt": row.get("prompt"),
            "chosen": row.get("chosen"),
            "rejected": row.get("rejected"),
        })

    source_meta = pd.DataFrame(source_rows).drop_duplicates("item_id")
    enriched = item_meta.copy()
    enriched["item_id"] = enriched["item_id"].astype(str)
    enriched = enriched.merge(source_meta, on="item_id", how="left")
    print(f"Enriched item metadata with prompt text for {enriched['prompt'].notna().sum()}/{len(enriched)} items")
    return enriched

if config.get("enrich_kudge"):
    item_meta = enrich_kudge_item_metadata(item_meta)

item_content_field = config.get("item_content_field")
if item_content_field in item_meta.columns:
    item_contents = list(item_meta[item_content_field].fillna("").astype(str))
else:
    item_contents = list(item_meta.iloc[:, 0].astype(str))

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=item_contents,
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": config["benchmark_id"],
        "item_id_field": config["item_id_field"],
        "value": config["value"],
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")


Loading mmlu_judging
../benchmarks/mmlu/response_matrices/mmlu_pro_judging_response_matrix.csv
10 models x 308 items, density = 99.7%


### K-Factor Fit

Fit `LogisticFM` with `K=1` and `K=2` on the response matrix. Rows stay as models/subjects and columns stay as items. Here `Z` is item easiness, so item difficulty is `-Z`.


In [74]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from torch_measure.models import LogisticFM, predict_dense
from torch_measure.models.rotation import varimax_rotation
from torch_measure.metrics.functional import compute_all

# Shared dense matrix + observed mask for all K-factor fits.
data = rm.data.to(device).float()
mask = ~torch.isnan(data) & (data != -1)

print(f"K-factor data: {data.shape[0]} models x {data.shape[1]} items, observed={mask.float().mean().item():.1%}")


K-factor data: 10 models x 308 items, observed=99.7%


In [75]:
def fit_logistic_fm_k(data, mask, k, device="cpu", max_epochs=1000, lr=0.03, seed=123):
    torch.manual_seed(seed + k)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed + k)

    model = LogisticFM(
        n_subjects=data.shape[0],
        n_items=data.shape[1],
        n_factors=k,
        device=device,
    )

    history = model.fit(
        data,
        mask=mask,
        method="mle",
        max_epochs=max_epochs,
        lr=lr,
        verbose=True,
    )

    with torch.no_grad():
        probs = predict_dense(model).detach()
    metrics = compute_all(probs, data, mask=mask)

    return {"model": model, "history": history, "metrics": metrics}


kfactor_specs = {
    1: {"max_epochs": 1000, "lr": 0.03},
    2: {"max_epochs": 1000, "lr": 0.02},
}

kfactor_results = {}
for k, spec in kfactor_specs.items():
    print(f"\nFitting LogisticFM K={k}")
    kfactor_results[k] = fit_logistic_fm_k(
        data,
        mask,
        k=k,
        device=device,
        max_epochs=spec["max_epochs"],
        lr=spec["lr"],
        seed=123,
    )

kfactor_fit_summary = pd.DataFrame([
    {
        "k": k,
        "loss": result["history"]["losses"][-1],
        "auc": result["metrics"].get("auc"),
        "log_likelihood": result["metrics"].get("log_likelihood"),
        "brier": result["metrics"].get("brier"),
        "ece": result["metrics"].get("ece"),
    }
    for k, result in kfactor_results.items()
])

display(kfactor_fit_summary.round(4))



Fitting LogisticFM K=1


MLE fitting: 100%|██████████| 1000/1000 [00:12<00:00, 79.61it/s, loss=0.230777]



Fitting LogisticFM K=2


MLE fitting: 100%|██████████| 1000/1000 [00:14<00:00, 68.84it/s, loss=0.115205]


,k,loss,auc,log_likelihood,brier,ece
0,1,0.2308,0.9463,-0.2308,0.0768,0.0298
1,2,0.1152,0.9876,-0.1152,0.0359,0.0169


### Item Difficulty With Laplace Uncertainty

For `LogisticFM`, `Z` is item easiness and `difficulty = -Z`. The uncertainty below uses a diagonal/conditional Laplace approximation: item factors and model factors are held fixed, and item-difficulty uncertainty is estimated from Bernoulli curvature `sum p(1-p)` over observed model responses.


In [76]:
def build_item_difficulty_table(result, rm, item_meta=None):
    model = result["model"]

    difficulty = model.difficulty.detach().cpu()
    difficulty_centered = difficulty - difficulty.mean()
    easiness_z = model.Z.detach().cpu()
    loadings = model.loadings.detach().cpu()

    # Rotate loadings for interpretability. For K=1 this is the identity.
    rotated_loadings, rotation = varimax_rotation(loadings)
    rotated_abilities = model.U.detach().cpu() @ rotation

    table = pd.DataFrame({
        "item_id": rm.item_ids,
        "difficulty": difficulty.numpy(),
        "difficulty_centered": difficulty_centered.numpy(),
        "easiness_z": easiness_z.numpy(),
    })

    for j in range(rotated_loadings.shape[1]):
        table[f"loading_factor_{j + 1}"] = rotated_loadings[:, j].numpy()

    loading_cols = [c for c in table.columns if c.startswith("loading_factor_")]
    table["dominant_factor"] = (
        table[loading_cols]
        .abs()
        .idxmax(axis=1)
        .str.replace("loading_factor_", "factor_", regex=False)
    )

    if item_meta is not None:
        table = table.merge(
            item_meta.reset_index(drop=True),
            left_index=True,
            right_index=True,
            how="left",
            suffixes=("", "_meta"),
        )

    model_factors = pd.DataFrame({"model": rm.subject_ids})
    for j in range(rotated_abilities.shape[1]):
        model_factors[f"factor_{j + 1}"] = rotated_abilities[:, j].numpy()

    return table, model_factors


item_difficulty_tables = {}
model_factor_tables = {}
for k, result in kfactor_results.items():
    item_table, model_table = build_item_difficulty_table(result, rm, item_meta=item_meta)
    item_difficulty_tables[k] = item_table
    model_factor_tables[k] = model_table

print("K=1 hardest items")
display(item_difficulty_tables[1].sort_values("difficulty_centered", ascending=False).head(20).round(3))

print("K=2 hardest items")
display(item_difficulty_tables[2].sort_values("difficulty_centered", ascending=False).head(20).round(3))


K=1 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,original_id,order,pair_id,source,split,response_model,gold_label
132,5087:original,16.114,17.177999,-16.114,14.842,factor_1,5087:original,5087,original,585e5a43-454b-56f7-aba3-fb4e22938020,mmlu-pro-other,gpt,gpt-4o-2024-05-13,B>A
94,4019:original,14.612,15.677000,-14.612,11.021,factor_1,4019:original,4019,original,129dfd5d-5786-57e3-8548-df0d26a42659,mmlu-pro-chemistry,gpt,gpt-4o-2024-05-13,B>A
186,7403:original,13.693,14.757000,-13.693,4.083,factor_1,7403:original,7403,original,07019bab-fc3e-5c91-ac9c-36869cfe79b0,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A
17,687:swapped,12.731,13.795000,-12.731,7.162,factor_1,687:swapped,687,swapped,6ad28b38-685f-5146-ab23-ff8fbcc81210,mmlu-pro-business,gpt,gpt-4o-2024-05-13,B>A
196,7623:original,12.605,13.669000,-12.605,7.133,factor_1,7623:original,7623,original,3c6f3a85-c058-543a-8226-852972410970,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A
291,11421:swapped,12.571,13.635000,-12.571,9.486,factor_1,11421:swapped,11421,swapped,bd2ab0d9-9fed-5c10-94a7-efa1750148f4,mmlu-pro-engineering,gpt,gpt-4o-2024-05-13,B>A
161,6105:swapped,12.436,13.500000,-12.436,7.042,factor_1,6105:swapped,6105,swapped,0cdc4e4b-d2a3-5218-b01a-c60f2e407d5e,mmlu-pro-health,gpt,gpt-4o-2024-05-13,B>A
166,6311:original,12.215,13.279000,-12.215,9.211,factor_1,6311:original,6311,original,d61248f3-8e80-5097-8892-10c086a492d2,mmlu-pro-health,gpt,gpt-4o-2024-05-13,B>A
172,6728:original,10.763,11.827000,-10.763,3.205,factor_1,6728:original,6728,original,0a347026-2ec5-5879-9635-e6fc818d021b,mmlu-pro-health,gpt,gpt-4o-2024-05-13,B>A
53,2416:swapped,10.659,11.723000,-10.659,3.182,factor_1,2416:swapped,2416,swapped,c20860f9-7468-546b-ae86-4ae6a6493de9,mmlu-pro-psychology,gpt,gpt-4o-2024-05-13,B>A


K=2 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,original_id,order,pair_id,source,split,response_model,gold_label
39,1748:swapped,15.002,15.226,-15.002,-3.190,1.713,factor_1,1748:swapped,1748,swapped,2d989dfb-7cf0-549e-945c-3dd060d1fad5,mmlu-pro-law,gpt,gpt-4o-2024-05-13,B>A
117,4789:swapped,12.894,13.118,-12.894,-0.281,4.546,factor_2,4789:swapped,4789,swapped,14ead85f-e4c6-5ae2-a586-6388d9b9b4f1,mmlu-pro-history,gpt,gpt-4o-2024-05-13,B>A
71,3036:swapped,11.202,11.426,-11.202,-4.990,2.378,factor_1,3036:swapped,3036,swapped,3cbbae47-8be1-5e31-868b-3b61d51a777d,mmlu-pro-biology,gpt,gpt-4o-2024-05-13,B>A
279,11220:swapped,10.934,11.158,-10.934,-2.178,2.819,factor_2,11220:swapped,11220,swapped,2f665bcd-7aee-539b-9acf-142548185b64,mmlu-pro-philosophy,gpt,gpt-4o-2024-05-13,B>A
189,7450:swapped,10.445,10.669,-10.445,-2.072,2.694,factor_2,7450:swapped,7450,swapped,f805817d-968d-5b0b-a2b2-4c62c9b848d3,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A
190,7568:original,10.368,10.591,-10.368,7.736,1.679,factor_1,7568:original,7568,original,a5c0fd0a-7440-5461-9a32-c6c5ff574ae0,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,A>B
244,10565:original,10.112,10.335,-10.112,-2.171,1.172,factor_1,10565:original,10565,original,07c3dda8-0f84-5624-9b4a-19ed31d85a2b,mmlu-pro-computer science,gpt,gpt-4o-2024-05-13,B>A
113,4771:swapped,9.917,10.140,-9.917,5.274,2.301,factor_1,4771:swapped,4771,swapped,2b51f48f-90d7-57f7-8888-30fd13c0d5ff,mmlu-pro-history,gpt,gpt-4o-2024-05-13,A>B
56,2453:original,9.785,10.009,-9.785,0.968,5.666,factor_2,2453:original,2453,original,9536d58f-8a88-53be-b97e-45bb2d457c12,mmlu-pro-psychology,gpt,gpt-4o-2024-05-13,A>B
41,1803:swapped,9.495,9.718,-9.495,-8.655,0.106,factor_1,1803:swapped,1803,swapped,01fb6121-e025-5251-a55f-f903c79e4ec6,mmlu-pro-law,gpt,gpt-4o-2024-05-13,B>A


In [77]:
def item_difficulty_laplace_se(model, data, mask):
    """Conditional diagonal Laplace SE for LogisticFM item difficulty.

    This treats fitted model factors/loadings as fixed and estimates curvature
    only for each item intercept/difficulty. For difficulty = -Z, the sign
    does not change the curvature.
    """
    data = data.to(model.device).float()
    mask = mask.to(model.device)

    with torch.no_grad():
        probs = predict_dense(model).clamp(1e-7, 1 - 1e-7)
        info = torch.where(mask, probs * (1 - probs), torch.zeros_like(probs)).sum(dim=0)
        raw_se = 1 / torch.sqrt(info.clamp_min(1e-8))

        # Our main reported difficulty is mean-centered. Under a diagonal
        # approximation, propagate uncertainty through d_j - mean(d).
        n_items = raw_se.numel()
        raw_var = raw_se.pow(2)
        centered_var = ((1 - 1 / n_items) ** 2) * raw_var + (raw_var.sum() - raw_var) / (n_items ** 2)
        centered_se = torch.sqrt(centered_var.clamp_min(1e-12))

    return raw_se.detach().cpu(), centered_se.detach().cpu()


item_difficulty_with_uncertainty = {}
for k, result in kfactor_results.items():
    raw_se, centered_se = item_difficulty_laplace_se(result["model"], data, mask)

    table = item_difficulty_tables[k].copy()
    table["difficulty_laplace_se"] = raw_se.numpy()
    table["difficulty_centered_laplace_se"] = centered_se.numpy()
    table["difficulty_centered_laplace_lo"] = table["difficulty_centered"] - 1.96 * table["difficulty_centered_laplace_se"]
    table["difficulty_centered_laplace_hi"] = table["difficulty_centered"] + 1.96 * table["difficulty_centered_laplace_se"]
    item_difficulty_with_uncertainty[k] = table.sort_values("difficulty_centered", ascending=False).reset_index(drop=True)

print("K=1 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[1].round(3))

print("K=2 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[2].round(3))


K=1 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,original_id,order,pair_id,source,split,response_model,gold_label,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,5087:original,16.114,17.177999,-16.114,14.842,factor_1,5087:original,5087,original,585e5a43-454b-56f7-aba3-fb4e22938020,mmlu-pro-other,gpt,gpt-4o-2024-05-13,B>A,1.843,1.856,13.541000,20.816000
1,4019:original,14.612,15.677000,-14.612,11.021,factor_1,4019:original,4019,original,129dfd5d-5786-57e3-8548-df0d26a42659,mmlu-pro-chemistry,gpt,gpt-4o-2024-05-13,B>A,1.735,1.749,12.248000,19.105000
2,7403:original,13.693,14.757000,-13.693,4.083,factor_1,7403:original,7403,original,07019bab-fc3e-5c91-ac9c-36869cfe79b0,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A,5.843,5.830,3.330000,26.183001
3,687:swapped,12.731,13.795000,-12.731,7.162,factor_1,687:swapped,687,swapped,6ad28b38-685f-5146-ab23-ff8fbcc81210,mmlu-pro-business,gpt,gpt-4o-2024-05-13,B>A,2.284,2.292,9.303000,18.287001
4,7623:original,12.605,13.669000,-12.605,7.133,factor_1,7623:original,7623,original,3c6f3a85-c058-543a-8226-852972410970,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A,2.268,2.276,9.208000,18.129999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,10565:swapped,-10.315,-9.250000,10.315,-4.666,factor_1,10565:swapped,10565,swapped,07c3dda8-0f84-5624-9b4a-19ed31d85a2b,mmlu-pro-computer science,gpt,gpt-4o-2024-05-13,A>B,1.372,1.393,-11.980000,-6.521000
304,11395:swapped,-10.844,-9.780000,10.844,-3.238,factor_1,11395:swapped,11395,swapped,83adf077-567f-5ce2-91e7-bf02decdaa21,mmlu-pro-engineering,gpt,gpt-4o-2024-05-13,A>B,3.622,3.619,-16.874001,-2.686000
305,5145:original,-11.718,-10.653000,11.718,-1.603,factor_1,5145:original,5145,original,b1614f17-39fb-564a-84c7-bcd7a2054dc8,mmlu-pro-other,gpt,gpt-4o-2024-05-13,A>B,2.292,2.299,-15.160000,-6.146000
306,7568:original,-12.011,-10.947000,12.011,-11.119,factor_1,7568:original,7568,original,a5c0fd0a-7440-5461-9a32-c6c5ff574ae0,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,A>B,1.625,1.641,-14.163000,-7.731000


K=2 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,original_id,order,pair_id,source,split,response_model,gold_label,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,1748:swapped,15.002,15.226,-15.002,-3.190,1.713,factor_1,1748:swapped,1748,swapped,2d989dfb-7cf0-549e-945c-3dd060d1fad5,mmlu-pro-law,gpt,gpt-4o-2024-05-13,B>A,5.826,5.813,3.833000,26.618000
1,4789:swapped,12.894,13.118,-12.894,-0.281,4.546,factor_2,4789:swapped,4789,swapped,14ead85f-e4c6-5ae2-a586-6388d9b9b4f1,mmlu-pro-history,gpt,gpt-4o-2024-05-13,B>A,1.642,1.656,9.873000,16.363001
2,3036:swapped,11.202,11.426,-11.202,-4.990,2.378,factor_1,3036:swapped,3036,swapped,3cbbae47-8be1-5e31-868b-3b61d51a777d,mmlu-pro-biology,gpt,gpt-4o-2024-05-13,B>A,3.592,3.589,4.391000,18.461000
3,11220:swapped,10.934,11.158,-10.934,-2.178,2.819,factor_2,11220:swapped,11220,swapped,2f665bcd-7aee-539b-9acf-142548185b64,mmlu-pro-philosophy,gpt,gpt-4o-2024-05-13,B>A,4.552,4.544,2.252000,20.063999
4,7450:swapped,10.445,10.669,-10.445,-2.072,2.694,factor_2,7450:swapped,7450,swapped,f805817d-968d-5b0b-a2b2-4c62c9b848d3,mmlu-pro-economics,gpt,gpt-4o-2024-05-13,B>A,4.135,4.129,2.575000,18.761999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,5284:swapped,-8.754,-8.530,8.754,0.110,-1.271,factor_2,5284:swapped,5284,swapped,ebe368cb-5783-5f03-8471-e31295bc4438,mmlu-pro-other,gpt,gpt-4o-2024-05-13,A>B,2.960,2.961,-14.333000,-2.727000
304,9337:swapped,-8.775,-8.552,8.775,4.036,-0.691,factor_1,9337:swapped,9337,swapped,df1964d5-629f-5b4a-bcd5-79137e8f3bc5,mmlu-pro-physics,gpt,gpt-4o-2024-05-13,A>B,1.956,1.965,-12.404000,-4.699000
305,9524:swapped,-8.858,-8.634,8.858,1.523,-6.453,factor_2,9524:swapped,9524,swapped,34e8d16b-9824-5373-b735-d25a3df21044,mmlu-pro-physics,gpt,gpt-4o-2024-05-13,A>B,3.636,3.633,-15.755000,-1.513000
306,8762:original,-10.051,-9.827,10.051,2.302,-0.699,factor_1,8762:original,8762,original,730909ae-6903-517a-ad99-9991d2cfc217,mmlu-pro-math,gpt,gpt-4o-2024-05-13,A>B,3.834,3.830,-17.333000,-2.321000


In [78]:
# Optional saves
result_name = KFACTOR_MATRIX
out_dir = Path("results") / result_name
out_dir.mkdir(parents=True, exist_ok=True)

def item_score_records(df, item_ids):
    """Return {item_id: {model: observed score}} with NaN converted to None."""
    score_map = {}
    for item_id in item_ids:
        values = df[str(item_id)] if str(item_id) in df.columns else df[item_id]
        score_map[str(item_id)] = {
            str(model): (None if pd.isna(value) else float(value))
            for model, value in values.items()
        }
    return score_map

scores_by_item = item_score_records(df, rm.item_ids)

kfactor_fit_summary.to_csv(out_dir / f"{result_name}_kfactor_fit_summary.csv", index=False)
item_difficulty_json = {
    "matrix": result_name,
    "benchmark_id": rm.info.get("benchmark_id"),
    "item_id_field": rm.info.get("item_id_field"),
    "value": rm.info.get("value"),
    "fits": {},
}
for k, table in item_difficulty_with_uncertainty.items():
    table.to_csv(out_dir / f"{result_name}_kfactor_k{k}_item_difficulties_with_laplace_uncertainty.csv", index=False)
    model_factor_tables[k].to_csv(out_dir / f"{result_name}_kfactor_k{k}_model_factors.csv", index=False)

    records = table.astype(object).where(pd.notna(table), None).to_dict("records")
    for record in records:
        record["scores"] = scores_by_item.get(str(record["item_id"]), {})
    item_difficulty_json["fits"][f"k{k}"] = records

with open(out_dir / f"{result_name}_kfactor_item_difficulties_with_laplace_uncertainty.json", "w") as f:
    json.dump(item_difficulty_json, f, indent=2)

print(f"Saved K-factor tables to {out_dir.resolve()}")


Saved K-factor tables to /Users/dkoffical/Documents/GitHub/cs321m_project/K-Factor/results/mmlu_judging


In [79]:
item_difficulty_json['fits']

{'k1': [{'item_id': '5087:original',
   'difficulty': 16.11383056640625,
   'difficulty_centered': 17.17820167541504,
   'easiness_z': -16.11383056640625,
   'loading_factor_1': 14.842421531677246,
   'dominant_factor': 'factor_1',
   'item_id_meta': '5087:original',
   'original_id': 5087,
   'order': 'original',
   'pair_id': '585e5a43-454b-56f7-aba3-fb4e22938020',
   'source': 'mmlu-pro-other',
   'split': 'gpt',
   'response_model': 'gpt-4o-2024-05-13',
   'gold_label': 'B>A',
   'difficulty_laplace_se': 1.8432888984680176,
   'difficulty_centered_laplace_se': 1.855886459350586,
   'difficulty_centered_laplace_lo': 13.540664672851562,
   'difficulty_centered_laplace_hi': 20.815738677978516,
   'scores': {'mistralai_ministral_3_14b_instruct_2512_bf16': 1.0,
    'mistralai_ministral_3_3b_instruct_2512_bf16': 1.0,
    'mistralai_ministral_3_8b_instruct_2512_bf16': 1.0,
    'qwen_qwen3_5_0_8b': 0.0,
    'qwen_qwen3_5_27b': 1.0,
    'qwen_qwen3_5_2b': 0.0,
    'qwen_qwen3_5_4b': 1.0,
  